# Correlation of HardCoded Triangle Proxies
We have shown before that the hardcoded triangle proxies can achieve pretty good approximation ratios. I would like to visualize the Pearson correlation between the proxy distribution $N(c';d,c)$ and that obtained by averaging over random Erdos-Renyi graphs with edge probability one-half. This should give us an idea of whether the correlation is closely related to the approximation ratio or not.

I am also interested in visualizing the sum of the corellation coefficients, weighted by $P(c)$.

## Imports and Function Definitions

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import qokit.maxcut as mc
import os
import grips
from grips import (
    get_simulator, QAOA_run, QAOA_proxy,
    QAOA_proxy_expectation, QAOA_proxy_optimize_gamma_beta,
    QAOA_proxy_expectation_from_gamma_beta,
    get_homogeneous_distribution,get_homogeneous_distribution_from_proxy,
    HardCodedTriangleProxy, TriangleProxy, NormalProxy, PaperProxy,
    inverse_objective_function, get_expectation,  
    maxcut, maxcut_approx_ratio, spsa_for_scipy,
    plot_distribution_lines_all, fit_proxy_to_real,
    get_pearson_correlation_coefficients,
    pad_and_stack, pad_to_shape
)

from QAOAKit import (                
    beta_to_qaoa_format,                 
    gamma_to_qaoa_format,                
    qaoa_maxcut_energy,                  
)                  

def mse_dist_loss(params, homodist, num_constraints=0):
    h_tweak_sub, hc_tweak_add, l_tweak_mul, r_tweak_mul = params
    num_constraints = max(homodist.shape[0] - 1, num_constraints)
    num_qubits = homodist.shape[1] - 1

    proxy = TriangleProxy(
        num_constraints=num_constraints,
        num_qubits=num_qubits,
        h_tweak_sub=h_tweak_sub,
        hc_tweak_add=hc_tweak_add,
        l_tweak_mul=l_tweak_mul,
        r_tweak_mul=r_tweak_mul
    )
    return grips.distribution_mean_squared_error(proxy, homodist)


def n_choose_2(n):
    return n * (n - 1) // 2

from juliacall import Main as jl
jl.seval('''
using Pkg
Pkg.activate(joinpath(@__DIR__, "../julia"))
Pkg.instantiate()
using JuliaQAOA
''')

# Choose Graph Pameters

In [ ]:
num_nodes = 8
edge_probability = 0.5
expected_num_constraints = int(n_choose_2(num_nodes)*edge_probability) # Used for proxies

# Graph and Homogeneous Distribution Setup
Set up several graphs, compute the same-cost-bitstring averaged homogeneous distribution, then take the average of those to get the homogeneous distribution for the whole class of graphs.

Question: is taking the average legitimate? Or do we need to sum up $n(x;d,c)$ over all graph instances and compute $N(c';d,c)$ from that?

In [ ]:
num_graphs = 25
graphs = [nx.erdos_renyi_graph(num_nodes, edge_probability, seed=i)
          for i in range(num_graphs)]
homo_distributions = [get_homogeneous_distribution(g) for g in graphs]
homo_distributions = pad_and_stack(homo_distributions)
sampled_homodist = np.mean(homo_distributions, axis=0)
sampled_shape = sampled_homodist.shape

plot_distribution_lines_all(sampled_homodist, "Averaged Homogeneous Distribution")

## Hardcoded Triangle Distribution

In [ ]:
hardcoded_proxy = HardCodedTriangleProxy(expected_num_constraints, num_nodes)

hardcoded_homodist = get_homogeneous_distribution_from_proxy(hardcoded_proxy)
hardcoded_homodist = pad_to_shape(hardcoded_homodist, sampled_shape)

plot_distribution_lines_all(hardcoded_homodist, "Hardcoded Triangle Homogeneous Distribution")


## Observations on the hardcoded triangle distribution
- The scale of each distribution is the same for every value of $c'. In the averged distribution, it seems to decay as $c'$ increases. This could be accomodated in the proxy. 
- Triangle proxy does seem to capture that the high distance distributions "move" to the right as $c'$ increases.

## Paper Proxy Distribution
Now we also compare the paper proxy.

In [ ]:
paper_proxy = PaperProxy(expected_num_constraints, num_nodes, edge_probability)
paper_homodist = get_homogeneous_distribution_from_proxy(paper_proxy)
paper_homodist = pad_to_shape(paper_homodist, sampled_shape)
plot_distribution_lines_all(paper_homodist, "Paper Proxy Homogeneous Proxy")

## Observations on Paper Proxy
- The scale seems to be off by a lot. Not just the constant scale, but the scale as you go between different values of $c'$. I wonder if there is a justification for this? I think it should matter, since if $N(c',:,:)$ is larger for some values of $c'$ compared to others, then the probability amplitudes of those $Q_\ell(c')$ should grow faster than others (according to the update rule (5)).

## Correlation Comparison
Now lets compare the Pearson correlation coefficients for each graph.

Note: the correlation coefficients are calculate for the 2D distributions $N(c';d,c)$ "raveled" into a 1D distribution. I'm a bit worried that that may not a good idea, and that something more intelligent should be done.

A good, but expensive way of comparing 2D distributions would be to use the Wasserstein distance from optimal transport. Try that if correlation does not work well.

In [ ]:
paper_pearson_coefficients = get_pearson_correlation_coefficients(sampled_homodist, paper_homodist)
hardcoded_pearson_coefficients = get_pearson_correlation_coefficients(sampled_homodist, hardcoded_homodist)

plt.plot(range(sampled_shape[0]), paper_pearson_coefficients, label="Paper Proxy", marker='o')
plt.plot(range(sampled_shape[0]), hardcoded_pearson_coefficients, label="Hardcoded Proxy", marker='o')
plt.ylim(0, 1.1)
plt.grid(True, which="both")
plt.xlabel("Cost (c') of Target Bitstring")
plt.ylabel("Pearson Correlation Coefficient")
plt.legend()
plt.show()

# Observations on Correlation
We are overall about as correlated as the Paper proxy. And we are even better correlated when $c'$ is large. That could be important, as that means our probability amplitudes $Q_\ell(c')$ will update more accurately for larger $c'$. On the other hand, if the lower $c'$ probability amplitudes update less accurately such that the become *larger*, that is bad for us.

## Approximation Ratio
1. Choose a random Erdos-Renyi graph using QAOAKit
2. Get the maximum cost using QOKit
3. Get optimal parameters using QAOAKit
4. Compute approximation ratios of Paper and Hardcoded Proxy
For this part, need to have QAOAKit data installed

In [ ]:
from QAOAKit.examples_utils import get_20_node_erdos_renyi_graphs
df = get_20_node_erdos_renyi_graphs()
# Get dataframe with for 10 instances of 20-node ER graphs, for p=1,2,3 QAOA layers (30 rows)
print(df.iloc[0])

In [ ]:
row = df.iloc[20]

G = row["G"]
p = row["p"]
assert p == 3, "Must have p=3 for this example"
optimized_cost = row['QAOA energy with optimized parameters']
optimal_cost = row['Optimal MaxCut solution from brute force']
#gammas = np.array(row["gamma"]) * np.pi
#betas = np.array(row["beta"]) * np.pi
gammas = gamma_to_qaoa_format(row["gamma"])
betas = beta_to_qaoa_format(row["beta"])
print("Number of edges: ", G.number_of_edges())
print(f"Gammas: {gammas}\nBetas: {betas}")
# Note that we want gamma to be pretty small for the proxies to work well

ising_model = mc.get_maxcut_terms(G)
sim = get_simulator(G.number_of_nodes(), ising_model)

## Compare optimal parameter schedule for proxies with analytic QAOA
Let's see if we can optimize the parameters using the proxies, then check the approximation ratios.

In [ ]:
paper_proxy_20 = jl.PaperProxy(G.number_of_edges(),G.number_of_nodes(), 0.5)

init_gamma = np.ones(p) * 0.2
init_beta =  np.ones(p) * 0.1
print("Initial Gammas: ", init_gamma)
print("Initial Betas: ", init_beta)

hardcoded_opt_dict = QAOA_proxy_optimize_gamma_beta(
    hardcoded_proxy_20, 
    init_gamma,
    init_beta,
    optimizer_method = "COBYLA",
    optimizer_options = {"disp": True},
    #optimizer_options = {"maxiter": 10, "disp": True},
)

hardcoded_true_expectation = get_expectation(
    G.number_of_nodes(), ising_model,
    hardcoded_opt_dict["gamma"], hardcoded_opt_dict["beta"],
    sim
)


#paper_opt_dict = QAOA_proxy_optimize_gamma_beta(
#    paper_proxy_20, 
#    init_gamma,
#    init_beta,
#    optimizer_method = "COBYLA",
#    optimizer_options = {"disp": True},
#    #optimizer_options = {"maxiter": 10, "disp": True},
#)


In [ ]:
hardcoded_proxy_20 = jl.HardCodedTriangleProxy(G.number_of_edges(), G.number_of_nodes())

paper_true_expectation = get_expectation(
    G.number_of_nodes(), ising_model,
    paper_opt_dict["gamma"], paper_opt_dict["beta"],
    sim
)

hardcoded_approximation_ratio = hardcoded_true_expectation / optimal_cost
paper_approximation_ratio = paper_true_expectation / optimal_cost
print("Paper Proxy Approximation Ratio: ", paper_approximation_ratio)
print("Hardcoded Proxy Approximation Ratio: ", hardcoded_approximation_ratio)

In [ ]:
sampled_homodist_20 = get_homogeneous_distribution(G)
hardcoded_homodist_20 = get_homogeneous_distribution_from_proxy(hardcoded_proxy_20)
paper_homodist_20 = get_homogeneous_distribution_from_proxy(paper_proxy_20)

### Extra Stuff, Ignore
Compute the cost expectation when using the proxies with the optimal analytic QAOA parameters.

Right now they don't look right, which leads me to believe that the problem is either that the gammas/betas are too big (and proxy works when they are small), or there is scaling that I am misunderstanding. To compare, I can try getting the expectation by actually simulating the QAOA circuit using QOKit with the optimal gamma/beta parameters.

EDIT: this test is actually a bad idea! The expectation scales for the proxies may be totally different than that of real QAOA! And although having the expectation be high if it is high for analytic QAOA is nice, the more important thing is that if the expectation is high for the proxies then it is also high for analytic QAOA.